In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Energy Market Price Forecasting - Initial Data Exploration\n",
    "## Exploratory Data Analysis of Electricity Prices\n",
    "\n",
    "**Author:** Amina Ali Hussein  \n",
    "**Date:** December 2024  \n",
    "**Status:** Initial exploration phase\n",
    "\n",
    "This notebook performs initial exploratory analysis on electricity price data to understand:\n",
    "- Price distributions and summary statistics\n",
    "- Temporal patterns (seasonality, trends)\n",
    "- Volatility characteristics\n",
    "- Potential relationships with exogenous variables"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import required libraries\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from datetime import datetime, timedelta\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set plotting style\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "sns.set_palette(\"husl\")\n",
    "%matplotlib inline"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Data Loading\n",
    "\n",
    "For this initial exploration, we'll create synthetic data that mimics real electricity price patterns.  \n",
    "**Next steps:** Replace with actual EIA data from https://www.eia.gov/electricity/"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate synthetic electricity price data for demonstration\n",
    "# This simulates realistic patterns we'd see in actual data\n",
    "\n",
    "np.random.seed(42)\n",
    "date_range = pd.date_range(start='2019-01-01', end='2024-12-01', freq='D')\n",
    "n_days = len(date_range)\n",
    "\n",
    "# Create base price with trend and seasonality\n",
    "base_price = 35  # $/MWh baseline\n",
    "trend = np.linspace(0, 10, n_days)  # Slight upward trend\n",
    "seasonality = 15 * np.sin(2 * np.pi * np.arange(n_days) / 365.25)  # Annual cycle\n",
    "weekly_pattern = 5 * np.sin(2 * np.pi * np.arange(n_days) / 7)  # Weekly cycle\n",
    "noise = np.random.normal(0, 8, n_days)  # Random volatility\n",
    "\n",
    "# Occasional price spikes (exogenous shocks)\n",
    "spikes = np.zeros(n_days)\n",
    "spike_indices = np.random.choice(n_days, size=int(n_days * 0.02), replace=False)\n",
    "spikes[spike_indices] = np.random.uniform(20, 80, len(spike_indices))\n",
    "\n",
    "# Combine components\n",
    "prices = base_price + trend + seasonality + weekly_pattern + noise + spikes\n",
    "prices = np.maximum(prices, 10)  # Floor at $10/MWh\n",
    "\n",
    "# Create DataFrame\n",
    "df = pd.DataFrame({\n",
    "    'date': date_range,\n",
    "    'price': prices\n",
    "})\n",
    "\n",
    "# Add derived features\n",
    "df['year'] = df['date'].dt.year\n",
    "df['month'] = df['date'].dt.month\n",
    "df['day_of_week'] = df['date'].dt.dayofweek\n",
    "df['quarter'] = df['date'].dt.quarter\n",
    "\n",
    "print(f\"Dataset shape: {df.shape}\")\n",
    "print(f\"Date range: {df['date'].min()} to {df['date'].max()}\")\n",
    "df.head(10)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Summary Statistics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Basic statistics\n",
    "print(\"=\" * 50)\n",
    "print(\"ELECTRICITY PRICE SUMMARY STATISTICS\")\n",
    "print(\"=\" * 50)\n",
    "print(df['price'].describe())\n",
    "print(f\"\\nCoefficient of Variation: {df['price'].std() / df['price'].mean():.2%}\")\n",
    "print(f\"Skewness: {df['price'].skew():.3f}\")\n",
    "print(f\"Kurtosis: {df['price'].kurtosis():.3f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Price Distribution Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "# Histogram\n",
    "axes[0].hist(df['price'], bins=50, edgecolor='black', alpha=0.7)\n",
    "axes[0].axvline(df['price'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: ${df[\"price\"].mean():.2f}')\n",
    "axes[0].axvline(df['price'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: ${df[\"price\"].median():.2f}')\n",
    "axes[0].set_xlabel('Price ($/MWh)', fontsize=12)\n",
    "axes[0].set_ylabel('Frequency', fontsize=12)\n",
    "axes[0].set_title('Distribution of Electricity Prices', fontsize=14, fontweight='bold')\n",
    "axes[0].legend()\n",
    "axes[0].grid(alpha=0.3)\n",
    "\n",
    "# Box plot by year\n",
    "df.boxplot(column='price', by='year', ax=axes[1])\n",
    "axes[1].set_xlabel('Year', fontsize=12)\n",
    "axes[1].set_ylabel('Price ($/MWh)', fontsize=12)\n",
    "axes[1].set_title('Price Distribution by Year', fontsize=14, fontweight='bold')\n",
    "plt.suptitle('')  # Remove automatic title\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Time Series Visualization"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(2, 1, figsize=(15, 10))\n",
    "\n",
    "# Full time series\n",
    "axes[0].plot(df['date'], df['price'], linewidth=1, alpha=0.7)\n",
    "axes[0].set_xlabel('Date', fontsize=12)\n",
    "axes[0].set_ylabel('Price ($/MWh)', fontsize=12)\n",
    "axes[0].set_title('Electricity Prices Over Time (2019-2024)', fontsize=14, fontweight='bold')\n",
    "axes[0].grid(alpha=0.3)\n",
    "\n",
    "# Zoom into recent period (last 6 months)\n",
    "recent_df = df[df['date'] >= '2024-06-01']\n",
    "axes[1].plot(recent_df['date'], recent_df['price'], linewidth=1.5, color='darkblue')\n",
    "axes[1].set_xlabel('Date', fontsize=12)\n",
    "axes[1].set_ylabel('Price ($/MWh)', fontsize=12)\n",
    "axes[1].set_title('Recent Price Movements (Last 6 Months)', fontsize=14, fontweight='bold')\n",
    "axes[1].grid(alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Seasonality Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Monthly average prices\n",
    "monthly_avg = df.groupby('month')['price'].mean().reset_index()\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(15, 5))\n",
    "\n",
    "# Monthly seasonality\n",
    "axes[0].bar(monthly_avg['month'], monthly_avg['price'], color='steelblue', edgecolor='black')\n",
    "axes[0].set_xlabel('Month', fontsize=12)\n",
    "axes[0].set_ylabel('Average Price ($/MWh)', fontsize=12)\n",
    "axes[0].set_title('Average Electricity Price by Month', fontsize=14, fontweight='bold')\n",
    "axes[0].set_xticks(range(1, 13))\n",
    "axes[0].set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])\n",
    "axes[0].grid(alpha=0.3, axis='y')\n",
    "\n",
    "# Day of week pattern\n",
    "dow_avg = df.groupby('day_of_week')['price'].mean().reset_index()\n",
    "axes[1].plot(dow_avg['day_of_week'], dow_avg['price'], marker='o', linewidth=2, markersize=8, color='darkgreen')\n",
    "axes[1].set_xlabel('Day of Week', fontsize=12)\n",
    "axes[1].set_ylabel('Average Price ($/MWh)', fontsize=12)\n",
    "axes[1].set_title('Average Price by Day of Week', fontsize=14, fontweight='bold')\n",
    "axes[1].set_xticks(range(7))\n",
    "axes[1].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])\n",
    "axes[1].grid(alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print(\"\\nObservations:\")\n",
    "print(f\"- Highest average price month: {monthly_avg.loc[monthly_avg['price'].idxmax(), 'month']}\")\n",
    "print(f\"- Lowest average price month: {monthly_avg.loc[monthly_avg['price'].idxmin(), 'month']}\")\n",
    "print(f\"- Peak-to-trough monthly difference: ${monthly_avg['price'].max() - monthly_avg['price'].min():.2f}/MWh\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Volatility Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Calculate rolling statistics\n",
    "df['rolling_mean_30d'] = df['price'].rolling(window=30).mean()\n",
    "df['rolling_std_30d'] = df['price'].rolling(window=30).std()\n",
    "\n",
    "fig, axes = plt.subplots(2, 1, figsize=(15, 10))\n",
    "\n",
    "# Price with rolling mean\n",
    "axes[0].plot(df['date'], df['price'], alpha=0.4, label='Daily Price', linewidth=0.8)\n",
    "axes[0].plot(df['date'], df['rolling_mean_30d'], color='red', linewidth=2, label='30-Day Rolling Mean')\n",
    "axes[0].set_xlabel('Date', fontsize=12)\n",
    "axes[0].set_ylabel('Price ($/MWh)', fontsize=12)\n",
    "axes[0].set_title('Price with 30-Day Rolling Average', fontsize=14, fontweight='bold')\n",
    "axes[0].legend()\n",
    "axes[0].grid(alpha=0.3)\n",
    "\n",
    "# Rolling volatility\n",
    "axes[1].plot(df['date'], df['rolling_std_30d'], color='darkred', linewidth=1.5)\n",
    "axes[1].fill_between(df['date'], 0, df['rolling_std_30d'], alpha=0.3, color='red')\n",
    "axes[1].set_xlabel('Date', fontsize=12)\n",
    "axes[1].set_ylabel('Standard Deviation ($/MWh)', fontsize=12)\n",
    "axes[1].set_title('30-Day Rolling Volatility', fontsize=14, fontweight='bold')\n",
    "axes[1].grid(alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Identifying Price Spikes (Potential Exogenous Shocks)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Define spike threshold (e.g., 2 standard deviations above mean)\n",
    "threshold = df['price'].mean() + 2 * df['price'].std()\n",
    "spikes_df = df[df['price'] > threshold].copy()\n",
    "\n",
    "print(f\"Price spike threshold: ${threshold:.2f}/MWh\")\n",
    "print(f\"Number of spike days: {len(spikes_df)} ({len(spikes_df)/len(df)*100:.1f}% of days)\")\n",
    "print(f\"\\nTop 10 highest price days:\")\n",
    "print(spikes_df.nlargest(10, 'price')[['date', 'price']])\n",
    "\n",
    "# Visualize spikes\n",
    "plt.figure(figsize=(15, 6))\n",
    "plt.plot(df['date'], df['price'], alpha=0.5, linewidth=1, label='Daily Price')\n",
    "plt.scatter(spikes_df['date'], spikes_df['price'], color='red', s=50, alpha=0.7, label='Price Spikes', zorder=5)\n",
    "plt.axhline(threshold, color='orange', linestyle='--', linewidth=2, label=f'Spike Threshold: ${threshold:.2f}')\n",
    "plt.xlabel('Date', fontsize=12)\n",
    "plt.ylabel('Price ($/MWh)', fontsize=12)\n",
    "plt.title('Electricity Prices with Identified Spikes', fontsize=14, fontweight='bold')\n",
    "plt.legend()\n",
    "plt.grid(alpha=0.3)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Next Steps\n",
    "\n",
    "### Immediate Actions:\n",
    "1. **Obtain real data** from EIA and NOAA\n",
    "2. **Integrate weather variables** (temperature, precipitation, wind speed)\n",
    "3. **Add natural gas prices** (strong correlation with electricity)\n",
    "4. **Create lagged features** for time series modeling\n",
    "\n",
    "### Modeling Pipeline:\n",
    "1. Train/test split (80/20, preserving temporal order)\n",
    "2. Baseline models (naive, ARIMA, SARIMAX)\n",
    "3. Machine learning models (XGBoost, Random Forest)\n",
    "4. Deep learning (LSTM for sequence modeling)\n",
    "5. Ensemble methods combining best performers\n",
    "\n",
    "### Analysis Questions to Address:\n",
    "- How do extreme weather events affect price volatility?\n",
    "- What is the lead time for predictable price movements?\n",
    "- Can we identify early warning signals for price spikes?\n",
    "- How do different forecasting horizons (1-day, 7-day, 30-day) compare in accuracy?"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "**Author:** Amina Ali Hussein  \n",
    "**Contact:** aminaali9816@gmail.com  \n",
    "**Last Updated:** December 2024"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}